In [ ]:
import os
import glob
import zipfile

from ipumspy import (
    AggregateDataExtract,
    IpumsApiClient,
    NhgisDataset,
    NhgisDatasetMetadata,
)


In [ ]:
ipums = IpumsApiClient("REDACTED")

download_dir = "./nhgis_data"
dataset = "2018_ACS1"


In [ ]:
tables = ipums.get_metadata(NhgisDatasetMetadata(dataset)).data_tables

In [13]:
print(tables[0].keys())

dict_keys(['name', 'description', 'universe', 'nhgisCode', 'sequence', 'datasetName', 'nVariables'])


In [ ]:
all_names = [table["name"] for table in tables]

In [15]:
for table in tables:
    name = table["name"]
    if name[-1].isalpha():
        # usually a race-based breakdown or puerto rico
        continue
    # skip columns with breakdown version
    if name[0] != "C" and "C" + name[1:] in all_names:
        continue
    print(
        f"{name:<10}{table['description'][:98]:<100}{table['universe'][:48]:<50}{table['nVariables']:>3}"
    )

B00001    Unweighted Sample Count of the Population                                                           Total population                                    1
B00002    Unweighted Sample Housing Units                                                                     Housing units                                       1
B01001    Sex by Age                                                                                          Total population                                   49
B01002    Median Age by Sex                                                                                   Total population                                    3
B01003    Total Population                                                                                    Total population                                    1
B02001    Race                                                                                                Total population                                   10
C02003    Detail

In [16]:
extract_tables = [
    "B01001",
    "B01003",
    "C03002",
    "C05001",
    "B23025",
    "B25002",
    "B25012",
    "B25064",
    "B25077",
    "B08137",
    "B08303",
    "C14002",
    "B20004",
]

In [ ]:
extract = AggregateDataExtract(
    collection="nhgis",
    description="all tables",
    datasets=[
        NhgisDataset(name=dataset, data_tables=extract_tables, geog_levels=["puma"])
    ],
    data_format="csv_header",
)

ipums.submit_extract(extract)

In [ ]:
ipums.wait_for_extract(extract)
ipums.download_extract(extract, download_dir="./nhgis_data")

In [26]:
for zip_path in glob.glob(f"{download_dir}/*.zip"):
    new_zip_path = os.path.join(download_dir, f"{dataset}.zip")
    os.rename(zip_path, new_zip_path)
    with zipfile.ZipFile(new_zip_path, "r") as z:
        z.extractall(download_dir)
    os.remove(new_zip_path)
